# Notebook 01: Warehouse Data Audit

This notebook performs Stage 1 Data Audit according to the Plan:
- Inspect raw CSV files, delimiters, and dimensions
- Check location distribution by vertical level $z$
- Audit catalog references against storage snapshot references
- Compute initial Mixing Index (MI) across scopes ($z=1, 2, 3, 4$ and $z \in \{3, 4\}$)

In [ ]:
import os
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from src.data.loaders import load_raw_csv
from src.data.cleaning import clean_storage_location, clean_product, clean_random_storage
from src.data.validation import validate_raw_datasets
from src.model.warehouse_state import WarehouseState

In [ ]:
# Load datasets
df_loc = load_raw_csv('Storage_Location.csv', raw_dir='../data/raw')
df_random = load_raw_csv('Random_Storage.csv', raw_dir='../data/raw')
df_prod = load_raw_csv('Product.csv', raw_dir='../data/raw')

audit_report = validate_raw_datasets(df_loc, df_random, df_prod)
print('Audit Report:')
for k, v in audit_report.items():
    print(f'  {k}: {v}')

In [ ]:
# Clean and normalize Random Storage with z scope in {3, 4}
long_df = clean_random_storage(df_random, df_locations=df_loc, z_filter=[3, 4])
print(f'Total slots in z in [3, 4]: {len(long_df)}')
print(f'Distinct locations: {long_df["originalLocation"].nunique()}')
print(f'Distinct References: {long_df["Reference"].nunique()}')
print(f'Observed size range: [{long_df["size"].min()}, {long_df["size"].max()}]')

In [ ]:
# Compute Baseline Mixing Index for z in {3, 4}
initial_state = WarehouseState.from_long_dataframe(long_df, capacity=18)
print(f'Baseline Average MI (z in [3, 4]): {initial_state.global_mixing_index():.4f}')
print(f'Critical Locations Rate (MI > 0.8): {initial_state.critical_locations_rate():.2f}%')